In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import TimestampedGeoJson
import pyarrow.parquet as pq
import os
import json
from io import StringIO
import branca.colormap as cm
import numpy as np
from datetime import datetime

# ==============================================================================
# GEOPTIMALISEERDE ANIMATIE MET TIMESTAMPEDGEOJSON
# ==============================================================================

# --- Configuratie ---
TOMTOM_FILE_PATH = os.path.join('Data', '20250820163000_stream.tomtom.analyze-sail.parquet')
SHAPEFILE_PATH = os.path.join('Data', 'NWB_roads', 'NWB_roads', 'wegen_in_out.shp')
SAMPLE_FRACTION = 0.5 # Behoud 50% sample
SHAPEFILE_ID_COL = 'wvk_id' # Correcte ID kolom
# OPTIMALISATIE: Aggregatie per 5 minuten
TIME_AGGREGATION = '5min'
# OPTIMALISATIE: Agressievere vereenvoudiging
SIMPLIFY_TOLERANCE = 0.0005


def parse_tomtom_data(row):
    """Verwerkt de data uit de '_value' kolom en voegt de timestamp toe."""
    value_string = row['_value']
    try:
        start_index = value_string.find('{')
        if start_index == -1: return None
        
        json_data = json.loads(value_string[start_index:])
        # Converteer tijd naar datetime object (timezone-naive for simplicity)
        timestamp = pd.to_datetime(json_data.get('time')).tz_localize(None)
        
        if 'data' in json_data and isinstance(json_data['data'], str):
            csv_data = pd.read_csv(StringIO(json_data['data']))
            if not csv_data.empty:
                csv_data['time'] = timestamp
                return csv_data
    except (json.JSONDecodeError, KeyError, ValueError, TypeError):
        return None
    return None

def create_animated_traffic_map():
    """Voert het volledige proces uit met TimestampedGeoJson en directe styling."""
    try:
        # --- 1. Data Laden, Verwerken en Aggregeren per Tijdseenheid ---
        print(f"Laden van {SAMPLE_FRACTION*100}% sample uit {TOMTOM_FILE_PATH}...")
        df_raw = pd.read_parquet(TOMTOM_FILE_PATH).sample(frac=SAMPLE_FRACTION, random_state=1)
        
        print("Verwerken van verkeersdata inclusief timestamps...")
        parsed_dfs = df_raw.apply(parse_tomtom_data, axis=1)
        df_traffic = pd.concat(parsed_dfs.dropna().tolist(), ignore_index=True)
        
        print(f"Aggregeren van data per wegvak per {TIME_AGGREGATION}...")
        df_traffic['time_bin'] = df_traffic['time'].dt.floor(TIME_AGGREGATION)
        df_agg = df_traffic.groupby(['id', 'time_bin'])['traffic_level'].mean().reset_index()
        print(f"Data geaggregeerd naar {len(df_agg):,} unieke tijd-wegvak combinaties.")

        # --- 2. Kaart Laden en Voorbereiden ---
        print(f"\nLaden van kaartdata: {SHAPEFILE_PATH}...")
        gdf = gpd.read_file(SHAPEFILE_PATH)
        gdf = gdf.to_crs(epsg=4326)
        
        print("Filteren op Amsterdam...")
        min_lon, min_lat, max_lon, max_lat = 4.72, 52.28, 5.08, 52.43
        gdf_amsterdam = gdf.clip([min_lon, min_lat, max_lon, max_lat]).copy()
        
        print(f"Agressief vereenvoudigen van geometrie (tolerance={SIMPLIFY_TOLERANCE})...")
        gdf_amsterdam['geometry'] = gdf_amsterdam.geometry.simplify(tolerance=SIMPLIFY_TOLERANCE, preserve_topology=True)

        # --- 3. Data Koppelen ---
        print("\nKoppelen van data...")
        gdf_amsterdam[SHAPEFILE_ID_COL] = pd.to_numeric(gdf_amsterdam[SHAPEFILE_ID_COL], errors='coerce').astype('Int64').astype(str)
        df_agg['id'] = pd.to_numeric(df_agg['id'], errors='coerce').astype('Int64').astype(str)
        df_agg['traffic_level'] = df_agg['traffic_level'].astype(float) # Zorg dat traffic level een float is
        
        # Gebruik 'inner' merge om alleen gematchte wegen te behouden
        merged_gdf = gdf_amsterdam.merge(df_agg, left_on=SHAPEFILE_ID_COL, right_on='id', how='inner')
        merged_gdf.dropna(subset=['traffic_level'], inplace=True) # Verwijder eventuele NaN na merge
        print(f"Succesvol {len(merged_gdf):,} tijd-wegvak combinaties gekoppeld.")

        if merged_gdf.empty:
            print("\nWAARSCHUWING: Geen data gekoppeld na merge.")
            return None

        # --- 4. Data Structureren voor TimestampedGeoJson ---
        print("Structureren van data voor de animatie...")
        colormap = cm.LinearColormap(colors=['green', 'yellow', 'red'], vmin=0, vmax=1)
        
        features = []
        # Group by time first for correct feature creation order
        merged_gdf['time_iso'] = merged_gdf['time_bin'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

        for _, row in merged_gdf.iterrows():
            level = row['traffic_level']
            # Handmatige controle of level geldig is voor de colormap
            if pd.isna(level):
                 color = '#808080' # Grijs voor NaN
                 opacity = 0.1
            else:
                 try:
                     color = colormap(float(level)) # Pas colormap toe
                     opacity = 0.8
                 except Exception: # Fallback bij onverwachte fouten
                     color = '#808080' # Grijs
                     opacity = 0.1
            
            # Maak het GeoJSON feature object
            feature = {
                'type': 'Feature',
                'geometry': row['geometry'].__geo_interface__,
                'properties': {
                    'time': row['time_iso'], # Gebruik ISO string tijd
                    'style': {              # Stijl per feature
                        'color': color,     # Lijnkleur
                        'weight': 2,        # Lijndikte
                        'opacity': opacity,
                        'fillColor': color, # Vulkleur (belangrijk voor lijnen)
                        'fillOpacity': opacity
                    },
                    'icon': 'circle', # Nodig voor plugin, ook voor lijnen
                    'iconstyle': {    # Maak het icoon onzichtbaar
                         'fillOpacity': 0,
                         'strokeOpacity': 0,
                         'radius': 0
                    },
                     # Tooltip data
                    'popup': f"ID: {row[SHAPEFILE_ID_COL]}<br>Drukte: {level:.2f}<br>Tijd: {row['time_bin'].strftime('%H:%M')}"
                }
            }
            features.append(feature)
        
        # --- 5. Visualisatie met TimestampedGeoJson ---
        print("Genereren van de geanimeerde Folium verkeerskaart...")
        
        amsterdam_location = [52.3676, 4.9041]
        m = folium.Map(location=amsterdam_location, zoom_start=12, tiles="CartoDB positron")

        TimestampedGeoJson(
            {'type': 'FeatureCollection', 'features': features},
            period=f'PT{TIME_AGGREGATION[-3:]}', # Periode instellen (bijv. 'PT5M')
            add_last_point=True,
            auto_play=False,
            loop=False,
            max_speed=5,
            loop_button=True,
            time_slider_drag_update=True,
            duration=f'PT{TIME_AGGREGATION[-3:]}'
        ).add_to(m)

        colormap.caption = f'Gemiddelde Verkeersdrukte (per {TIME_AGGREGATION})'
        m.add_child(colormap)
        
        print("\n--- SUCCES! ---")
        
        m.save("verkeerskaart_amsterdam_animated_final.html")
        print("Geoptimaliseerde geanimeerde kaart opgeslagen als 'verkeerskaart_amsterdam_animated_final.html'")
        
        return m

    except Exception as e:
        print(f"Er is een onverwachte fout opgetreden: {e}")
        import traceback
        traceback.print_exc()
        return None

# Voer het script uit
traffic_map = create_animated_traffic_map()
if traffic_map:
    from IPython.display import display 
    display(traffic_map)




Laden van 50.0% sample uit Data\20250820163000_stream.tomtom.analyze-sail.parquet...
Verwerken van verkeersdata inclusief timestamps...
Aggregeren van data per wegvak per 5min...
Data geaggregeerd naar 547,842 unieke tijd-wegvak combinaties.

Laden van kaartdata: Data\NWB_roads\NWB_roads\wegen_in_out.shp...
Filteren op Amsterdam...
Agressief vereenvoudigen van geometrie (tolerance=0.0005)...

Koppelen van data...
Succesvol 489,753 tijd-wegvak combinaties gekoppeld.
Structureren van data voor de animatie...
Genereren van de geanimeerde Folium verkeerskaart...

--- SUCCES! ---
Geoptimaliseerde geanimeerde kaart opgeslagen als 'verkeerskaart_amsterdam_animated_final.html'
